In [1]:
pip install plotly


  Using cached narwhals-2.15.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.9 MB 4.8 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.9 MB 4.2 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.9 MB 4.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 4.1 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/9.9 MB 4.2 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/9.9 MB 4.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.9 MB 3.3 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 3.4 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/9.9 MB 3.2 MB/s eta 0:00:02
   ---------------------------- ----------- 7.1/9.9 MB 3.2 MB/s eta 0:00:01
   ----------------------------- ----

In [2]:
pip install kaleido 


   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---- ----------------------------------- 1/9 [pluggy]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   -------------------------- ------------- 6/9 [choreographer]
   -------------------------- ------------- 6/9 [choreographer]
   -------------------------- ------------- 6/9 [choreographer]
   ----------------------------------- ---- 8/9 [kaleido]
   ----------------------------------- ---- 8/9 [kaleido]
   ----------------------------------- ---- 8/9 [kaleido]
   ---------------------------------------- 9/9 [kaleid

In [1]:
# model_working.ipynb

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Machine Learning imports
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import joblib

# Check GPU
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Set paths
train_dir = 'Rice_Leaf_Diease/train'
test_dir = 'Rice_Leaf_Diease/test'

# Check class names
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)
print(f"Classes: {num_classes}")
print(f"Class names: {class_names}")

# ==================================================
# DATA LOADING WITH GENERATORS (PROPER WAY)
# ==================================================

IMG_SIZE = 128  # Reduced but reasonable
BATCH_SIZE = 128

print(f"\n📊 Using ImageDataGenerator for proper data loading")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")

# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nData sizes:")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Testing samples: {test_generator.samples}")

# ==================================================
# CNN MODEL FOR FEATURE EXTRACTION
# ==================================================

print("\n" + "="*60)
print("BUILDING CNN MODEL")
print("="*60)

def create_cnn_model(input_shape=(128, 128, 3), num_classes=10):
    """Create a proper CNN model"""
    model = Sequential([
        # First conv block
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.2),
        
        # Second conv block
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),
        
        # Third conv block
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.4),
        
        # Global pooling and dense layers
        GlobalAveragePooling2D(),
        
        # Feature layer for SVM
        Dense(256, activation='relu', name='features'),
        BatchNormalization(),
        Dropout(0.5),
        
        # Classification layer
        Dense(num_classes, activation='softmax', name='classification')
    ])
    
    return model

# Create and compile model
cnn_model = create_cnn_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=num_classes)

cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

# Callbacks for better training
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

# ==================================================
# TRAIN CNN
# ==================================================

print("\n" + "="*60)
print("TRAINING CNN")
print("="*60)

# Calculate steps per epoch
train_steps = train_generator.samples // BATCH_SIZE
val_steps = val_generator.samples // BATCH_SIZE

print(f"Training steps per epoch: {train_steps}")
print(f"Validation steps per epoch: {val_steps}")

# Train the model
history = cnn_model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=val_generator,
    validation_steps=val_steps,
    epochs=15,  # Reasonable number of epochs
    callbacks=callbacks,
    verbose=1
)

# Show final accuracy
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\n✅ CNN Training Complete!")
print(f"Final Training Accuracy: {final_train_acc:.3f}")
print(f"Final Validation Accuracy: {final_val_acc:.3f}")

# ==================================================
# EXTRACT FEATURES
# ==================================================

print("\n" + "="*60)
print("EXTRACTING FEATURES")
print("="*60)

# Create feature extractor
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer('features').output
)

# Function to extract features from generator
def extract_features_from_generator(model, generator):
    """Extract features from all images in generator"""
    generator.reset()
    features = []
    labels = []
    
    for i in range(len(generator)):
        batch_images, batch_labels = generator[i]
        batch_features = model.predict(batch_images, verbose=0)
        features.append(batch_features)
        labels.append(batch_labels)
        
        # Break if we have all samples
        if len(features) * BATCH_SIZE >= generator.samples:
            break
    
    # Concatenate all batches
    features = np.vstack(features)[:generator.samples]
    labels = np.vstack(labels)[:generator.samples]
    labels = np.argmax(labels, axis=1)  # Convert one-hot to categorical
    
    return features, labels

# Extract features from all sets
print("Extracting training features...")
X_train_features, y_train = extract_features_from_generator(feature_extractor, train_generator)

print("Extracting validation features...")
X_val_features, y_val = extract_features_from_generator(feature_extractor, val_generator)

print("Extracting test features...")
X_test_features, y_test = extract_features_from_generator(feature_extractor, test_generator)

print(f"\nFeature shapes:")
print(f"Training: {X_train_features.shape}")
print(f"Validation: {X_val_features.shape}")
print(f"Testing: {X_test_features.shape}")

# ==================================================
# PCA FOR DIMENSIONALITY REDUCTION
# ==================================================

print("\n" + "="*60)
print("APPLYING PCA")
print("="*60)

# Use PCA to reduce dimensions
n_components = min(50, X_train_features.shape[1])
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_features)
X_val_pca = pca.transform(X_val_features)
X_test_pca = pca.transform(X_test_features)

print(f"PCA components: {n_components}")
print(f"Explained variance: {np.sum(pca.explained_variance_ratio_):.3f}")

# ==================================================
# SVM TRAINING
# ==================================================

print("\n" + "="*60)
print("TRAINING SVM CLASSIFIER")
print("="*60)

# Train SVM on combined train + validation features
X_combined = np.vstack([X_train_pca, X_val_pca])
y_combined = np.concatenate([y_train, y_val])

print(f"Training SVM on {len(X_combined)} samples...")
svm = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm.fit(X_combined, y_combined)

# Make predictions
y_test_pred = svm.predict(X_test_pca)
test_accuracy = accuracy_score(y_test, y_test_pred)
y_test_proba = svm.predict_proba(X_test_pca)

print(f"\n✅ SVM Training Complete!")
print(f"Test Accuracy: {test_accuracy:.3f}")
print(f"Test Samples: {len(y_test)}")

# ==================================================
# EVALUATION
# ==================================================

print("\n" + "="*60)
print("COMPREHENSIVE EVALUATION")
print("="*60)

# Classification report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=class_names, digits=3))

# Confusion matrix analysis
print("\n📈 Confusion Matrix Analysis:")
cm = confusion_matrix(y_test, y_test_pred)

for i, cls in enumerate(class_names):
    if i < len(cm):
        correct = cm[i, i]
        total = cm[i, :].sum()
        accuracy = correct / total if total > 0 else 0
        print(f"{cls:<25}: {correct:3d}/{total:3d} = {accuracy:.3f}")

# Overall metrics
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')
f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\n🎯 Overall Metrics:")
print(f"Accuracy:  {test_accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

# ==================================================
# SAVE MODELS AND RESULTS
# ==================================================

print("\n" + "="*60)
print("SAVING MODELS AND RESULTS")
print("="*60)

# Save predictions with details
results_df = pd.DataFrame({
    'True_Class': [class_names[i] for i in y_test],
    'Predicted_Class': [class_names[i] for i in y_test_pred],
    'Confidence': np.max(y_test_proba, axis=1),
    'Correct': y_test == y_test_pred
})

# Add all probabilities
for i, cls in enumerate(class_names):
    results_df[f'Prob_{cls}'] = y_test_proba[:, i]

# Save to CSV
results_df.to_csv('cnn_svm_predictions.csv', index=False)
print("✓ cnn_svm_predictions.csv - All predictions with probabilities")

# Save model metrics
metrics_dict = {
    'CNN_Train_Accuracy': final_train_acc,
    'CNN_Val_Accuracy': final_val_acc,
    'SVM_Test_Accuracy': test_accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1_Score': f1,
    'Training_Samples': train_generator.samples,
    'Test_Samples': test_generator.samples
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv('model_metrics.csv', index=False)
print("✓ model_metrics.csv - Performance metrics")

# Save models
joblib.dump(svm, 'trained_svm.pkl')
joblib.dump(pca, 'pca_transformer.pkl')
cnn_model.save('trained_cnn.h5')
feature_extractor.save('feature_extractor.h5')

print("✓ trained_svm.pkl - SVM model")
print("✓ pca_transformer.pkl - PCA transformer")
print("✓ trained_cnn.h5 - Full CNN model")
print("✓ feature_extractor.h5 - Feature extractor only")

# ==================================================
# PREDICTION FUNCTION
# ==================================================

def predict_rice_disease(image_path):
    """Predict rice leaf disease from image"""
    try:
        # Load and preprocess image
        img = tf.keras.preprocessing.image.load_img(
            image_path, target_size=(IMG_SIZE, IMG_SIZE)
        )
        img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        # Extract features
        features = feature_extractor.predict(img_array, verbose=0)
        
        # Apply PCA
        features_pca = pca.transform(features)
        
        # Predict
        prediction = svm.predict(features_pca)[0]
        probabilities = svm.predict_proba(features_pca)[0]
        
        predicted_class = class_names[prediction]
        confidence = probabilities[prediction]
        
        # Get top 3 predictions
        top_indices = np.argsort(probabilities)[-3:][::-1]
        top_predictions = [
            (class_names[idx], probabilities[idx]) 
            for idx in top_indices
        ]
        
        return {
            'predicted_class': predicted_class,
            'confidence': confidence,
            'all_probabilities': probabilities,
            'top_predictions': top_predictions
        }
        
    except Exception as e:
        print(f"Error predicting image: {e}")
        return None

# Test prediction with a real image
print("\n" + "="*60)
print("TESTING PREDICTION")
print("="*60)

# Find a real test image
test_image_path = None
for cls in class_names:
    cls_dir = os.path.join(test_dir, cls)
    if os.path.exists(cls_dir):
        image_files = [f for f in os.listdir(cls_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if image_files:
            test_image_path = os.path.join(cls_dir, image_files[0])
            true_class = cls
            break

if test_image_path:
    print(f"Testing on: {test_image_path}")
    print(f"True class: {true_class}")
    
    result = predict_rice_disease(test_image_path)
    
    if result:
        print(f"\n🔍 Prediction Results:")
        print(f"Predicted: {result['predicted_class']}")
        print(f"Confidence: {result['confidence']:.3f}")
        print(f"Correct: {result['predicted_class'] == true_class}")
        
        print(f"\n🏆 Top 3 Predictions:")
        for cls, prob in result['top_predictions']:
            print(f"  {cls}: {prob:.3f}")
else:
    print("No test image found for prediction demo")

# ==================================================
# SUMMARY
# ==================================================

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f" CNN-SVM Hybrid Model Training Complete!")
print(f"\n Dataset Statistics:")
print(f"  Classes: {num_classes}")
print(f"  Training samples: {train_generator.samples}")
print(f"  Validation samples: {val_generator.samples}")
print(f"  Testing samples: {test_generator.samples}")
print(f"\n Final Performance:")
print(f"  CNN Validation Accuracy: {final_val_acc:.3f}")
print(f"  SVM Test Accuracy: {test_accuracy:.3f}")
print(f"  Overall F1-Score: {f1:.3f}")

print("\n" + "="*60)
print(" ALL DONE! Ready for deployment.")
print("="*60)

GPU Available: True
Classes: 10
Class names: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 'sheath_blight', 'tungro']

📊 Using ImageDataGenerator for proper data loading
  Image size: 128x128
  Batch size: 128
Found 12020 images belonging to 10 classes.
Found 3003 images belonging to 10 classes.
Found 3421 images belonging to 10 classes.

Data sizes:
Training samples: 12020
Validation samples: 3003
Testing samples: 3421

BUILDING CNN MODEL
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 128, 128, 32)     128       
 ormalization)                                                   
                                                          

In [5]:
pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)

   ------------- -------------------------- 2/6 [referencing]
   -------------------- ------------------- 3/6 [jsonschema-specifications]
   -------------------------- ------------- 4/6 [jsonschema]
   -------------------------- ------------- 4/6 [jsonschema]
   --------------------------------- ------ 5/6 [nbformat]
   -----------------

In [2]:
# hybrid_prediction.ipynb

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib

# Load ALL trained models
cnn_model = load_model('trained_cnn.h5')
feature_extractor = load_model('feature_extractor.h5')
pca = joblib.load('pca_transformer.pkl')
svm = joblib.load('trained_svm.pkl')

# Class names (must match training)
class_names = [
    'bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 
    'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 
    'sheath_blight', 'tungro'
]

# Image size (must match training)
IMG_SIZE = 128

def predict_hybrid(image_path):
    """
    Predict using CNN-SVM hybrid model
    Returns: disease name, confidence, all probabilities
    """
    # 1. Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # 2. Extract features using CNN
    features = feature_extractor.predict(img_array, verbose=0)
    
    # 3. Apply PCA transformation
    features_pca = pca.transform(features)
    
    # 4. Get SVM prediction
    svm_prediction = svm.predict(features_pca)[0]
    svm_probabilities = svm.predict_proba(features_pca)[0]
    
    predicted_class = class_names[svm_prediction]
    confidence = svm_probabilities[svm_prediction]
    
    return predicted_class, confidence, svm_probabilities

# ========== USAGE ==========
# Replace with your image path
image_path = r"Rice_Leaf_Diease\train\leaf_scald\leaf_scald1662.jpg"

# Make prediction
disease, confidence, all_probs = predict_hybrid(image_path)

print(f"🎯 CNN-SVM Hybrid Prediction")
print(f"🔬 Disease: {disease}")
print(f"📈 Confidence: {confidence:.2%}")

# Show top 3 predictions
top_3_idx = np.argsort(all_probs)[-3:][::-1]
print("\n🏆 Top 3 Predictions:")
for idx in top_3_idx:
    print(f"  {class_names[idx]}: {all_probs[idx]:.2%}")

# Optional: Show all probabilities
print("\n📊 All Probabilities:")
for i, cls in enumerate(class_names):
    print(f"  {cls:<25}: {all_probs[i]:.4%}")

🎯 CNN-SVM Hybrid Prediction
🔬 Disease: leaf_scald
📈 Confidence: 98.61%

🏆 Top 3 Predictions:
  leaf_scald: 98.61%
  bacterial_leaf_blight: 0.92%
  narrow_brown_spot: 0.18%

📊 All Probabilities:
  bacterial_leaf_blight    : 0.9212%
  brown_spot               : 0.0177%
  healthy                  : 0.0144%
  leaf_blast               : 0.0364%
  leaf_scald               : 98.6062%
  narrow_brown_spot        : 0.1834%
  neck_blast               : 0.0348%
  rice_hispa               : 0.0072%
  sheath_blight            : 0.0111%
  tungro                   : 0.1677%
